### This is a trial to run the meta-learning agent in the trial environment
Population weights: Uniform

In [ ]:
import os, sys
os.environ["PYTHONPATH"] = "/home/rifnyrachman7" # change into real directory
sys.path.append("/home/rifnyrachman7/_merlion/")
sys.path.append("/home/rifnyrachman7/_merlion/ds-project-messiah/src")
sys.path.append("/home/rifnyrachman7/")

In [ ]:
from gymnasium.wrappers import TimeLimit
import gymnasium as gym
from pathlib import Path

# Import both modules
from _metamorl.algo_ray.rllib.algorithms.maml.merlion_main_multitasks import MERLION, MERLIONConfig
from _metamorl.algo_ray.rllib.algorithms.maml.callback_save_population import _find_archive , MERLIONWithSaveCallbacks

import ray # use version 2.3.1
from ray import air, tune
from ray.tune.registry import register_env
from gymnasium.wrappers import TimeLimit
from ray.air import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

# Import the required modules

from messiah.environment import Environment # alternatively use https://github.com/hubbs5/or-gym*
from messiah.agents.processing import MaxAvailableProcessingAgent

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

import tempfile
import pickle
import json


#from utils.plot_utils import PlotUtils

*@misc{HubbsOR-Gym,
    author={Christian D. Hubbs and Hector D. Perez and Owais Sarwar and Nikolaos V. Sahinidis and Ignacio E. Grossmann and John M. Wassick},
    title={OR-Gym: A Reinforcement Learning Library for Operations Research Problems},
    year={2020},
    Eprint={arXiv:2008.06319}
}

In [ ]:
#import generated environment
from _merlion.random_env import complex_env_merlion # change for other environment

In [ ]:
def make_env(env_config=None):
    base = ComplexSC()
    base = ScalarizationWeightProxy(base)
    base = TimeLimit(base, max_episode_steps=100)   # limit time step at a period
    return base

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# archive management
def save_full_checkpoints_for_population(algo, out_dir: str):
    archive = _find_archive(algo)
    if archive is None:
        print("[SavePopulation] MERLION archive not found on algorithm; skipping.")
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    pol = algo.get_policy()
    P = int(getattr(archive, "population_size", len(getattr(archive, "archive", []))))

    for i in range(P):
        item = archive.get_archive_item(i)
        if item is None:
            continue

        theta_i = item["theta"]
        w_i = np.asarray(item["weight"], dtype=np.float32)

        # save the full RLlib checkpoint for policy i
        pol.set_weights(theta_i)
        ckpt_dir = out / f"checkpoint_policy_{i}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        path = algo.save(str(ckpt_dir))
        print(f"[SavePopulation] Saved RLlib checkpoint for i={i} → {path}")

        # save the associated weight next to it
        with open(ckpt_dir / "weight.json", "w") as f:
            json.dump([float(x) for x in w_i.ravel().tolist()], f)

In [ ]:
ray.shutdown()
ray.init() # initiate ray

register_env("complex_sc", lambda cfg: complex_env_merlion.make_env(cfg)) # register environment

rollout_fragment_length = 32

### Adaptation and Metalearning Phase

In [ ]:
# set MERLION configuration
seed_base = 99

config = (
    MERLIONConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
    )
    .framework("torch")
    .environment("complex_sc", clip_actions=True)
    .training(
        inner_adaptation_steps=8,
        meta_batch_size=4, # follow rollout_workers in MIRACL that represent task batch number
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
        
        #MERLION specific parameters
        population_size = 21,
        num_objectives = 3,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
    .callbacks(MERLIONWithSaveCallbacks)
)

In [ ]:
local_dir = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0" # change to correct directory
num_iterations = 13

results = {}
runs = 10 # number of generated seeds

# run the meta-training
for k in runs:
    print(f"RUNNING NUMBER-{k}")
    
    tuner = tune.Tuner(
        MERLION,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            name=f"merlion_complex_{k}",
            local_dir = local_dir,
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),
            log_to_file=True,
            checkpoint_config=air.CheckpointConfig(checkpoint_at_end=True, num_to_keep=1),  # ensure a checkpoint exists
        )
    )
    results[k] = tuner.fit()
    
    # #save the best checkpoint for each run
    best = results[k].get_best_result()
    algo = Algorithm.from_checkpoint(best.checkpoint)
    
    out_dir = os.path.join(local_dir, f"metapolicies-{k}") # export trained meta-policies
    os.makedirs(out_dir, exist_ok=True)
    save_full_checkpoints_for_population(algo, out_dir)

### Fine-tune using stable-baselines3

In [ ]:
# import necessary modules
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from _merlion.fine_tuning.ft_complex_env_merlion import TestComplexSC
from _metamorl.algo_ray.rllib.algorithms.maml import merlion_finetuning_sb3_independent as ft

In [ ]:
# (Re)start Ray cleanly
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

def make_env_ray(config):
    """
    RLlib factory: receives a dict `config` and must return a *Gymnasium Env instance*.
    We wrap with TimeLimit here so both RLlib and Gym share the same horizon behavior.
    """
    base = TestComplexSC(**config)            # pass config keys into your env
    max_steps = int(config.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

register_env("testSC", make_env_ray)
register_env("complex_sc", lambda cfg: complex_env_merlion.make_env(cfg)) # register meta-env

In [ ]:
# Optional: re-register safely if re-running cells
try:
    gym.spec("testSC")  # raises if not registered
    # Gymnasium 0.29.x: remove existing spec to allow re-register
    del gym.envs.registration.registry["testSC"]
except Exception:
    pass

def make_env_gym(**kwargs):
    base = TestComplexSC(**kwargs)
    max_steps = int(kwargs.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id="testSC",
    entry_point=make_env_gym,   # callable that returns an Env
)

In [ ]:
# run fine-tuning
runs = 10 # number of generated seeds
for k in runs:
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/metapolicies-{k}", # meta-policy directory
        env_id="testSC",
        out_dir=f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-{k}", # fine-tuning directory
        total_steps=20000,
        record_every=512,
        eval_episodes=50,
        m_perturb=2,
        eps=0.05,
        seed=7, # deactivate for multiple runs
        skip_existing=True,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

### Load the model to print policy

In [ ]:
MODEL_PATH = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-0/i016_m01/sb3_model.zip"  # <- your file
ENV_ID     = "testSC"
CSV_OUT    = "/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-0/i016_m01/action_plan.csv"

# 1) Check the env spaces (what you're creating right now)
env = gym.make(ENV_ID, max_steps=100)
print("ENV observation_space:", env.observation_space)
print("ENV action_space     :", env.action_space)

obs, info = env.reset()
print("ENV reset obs shape  :", getattr(obs, "shape", type(obs)))

# 2) Check the model spaces (what it was trained with) WITHOUT attaching an env
model = PPO.load(MODEL_PATH, env=None)
print("MODEL observation_space:", model.observation_space)
print("MODEL action_space     :", model.action_space)

In [ ]:
run = 9
MODEL_PATH = f"/home/rifnyrachman7/_merlion/training_data/complex_sc_v0/finetuning_sb3-{run}/i009_m01/sb3_model.zip"  # <- your file
ENV_ID     = "testSC"
CSV_OUT    = f"/home/rifnyrachman7/_merlion/visualisation/action_com_merlion-{run}.csv"

env = gym.make(ENV_ID, max_steps=100)
model = PPO.load(MODEL_PATH, env=env)

# Try to obtain human-readable action labels, otherwise use a[i]
action_labels = None
ue = getattr(env, "unwrapped", env)
for attr in ("get_action_names", "action_names", "action_labels"):
    if hasattr(ue, attr):
        names = getattr(ue, attr)
        action_labels = names() if callable(names) else names
        break

obs, info = env.reset(seed=7)
rows = []
t = 0
while True:
    action, _ = model.predict(obs, deterministic=True)
    action = np.array(action).ravel()

    row = {"t": t}
    if action_labels and len(action_labels) == action.size:
        for name, val in zip(action_labels, action):
            row[name] = float(val)
    else:
        for i, val in enumerate(action):
            row[f"a{i}"] = float(val)

    # Optional: capture reward vector if the env provides it
    mo = info.get("mo_reward")
    if mo is not None and len(mo) >= 3:
        row["profit"]    = float(mo[0])
        row["emission"]  = float(mo[1])
        row["inequality"]= float(mo[2])

    rows.append(row)

    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    t += 1

env.close()

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print(f"Saved action plan to {CSV_OUT}")